In [41]:
import pandas as pd
import matplotlib.pyplot as plt

train = pd.read_csv('../data/raw/train.csv', parse_dates = ['Date'], low_memory = False)
store = pd.read_csv('../data/raw/store.csv')

df = train.merge(store, on='Store', how='left') # merge the two tables 
print(df.shape) #show the number of columns and rows
df.head()


(1017209, 18)


,Store,DayOfWeek,Date,Sales,Customers,Open,Promo,StateHoliday,SchoolHoliday,StoreType,Assortment,CompetitionDistance,CompetitionOpenSinceMonth,CompetitionOpenSinceYear,Promo2,Promo2SinceWeek,Promo2SinceYear,PromoInterval
0,1,5,2015-07-31,5263,555,1,1,0,1,c,a,1270.0,9.0,2008.0,0,NaN,NaN,NaN
1,2,5,2015-07-31,6064,625,1,1,0,1,a,a,570.0,11.0,2007.0,1,13.0,2010.0,"Jan,Apr,Jul,Oct"
2,3,5,2015-07-31,8314,821,1,1,0,1,a,a,14130.0,12.0,2006.0,1,14.0,2011.0,"Jan,Apr,Jul,Oct"
3,4,5,2015-07-31,13995,1498,1,1,0,1,c,c,620.0,9.0,2009.0,0,NaN,NaN,NaN
4,5,5,2015-07-31,4822,559,1,1,0,1,a,a,29910.0,4.0,2015.0,0,NaN,NaN,NaN


Finding out how many stores participate in the promo, this will also give us the time window

In [42]:
# How many stores ever participate in Promo2?
print(store['Promo2'].value_counts())
print()

# For participating stores, when did they join?
participants = store[store['Promo2'] == 1].copy()
print("Promo2 start years:")
print(participants['Promo2SinceYear'].value_counts().sort_index())
print()
print("Number of distinct (year, week) start points:",
      participants.groupby(['Promo2SinceYear', 'Promo2SinceWeek']).ngroups)

Promo2
1    571
0    544
Name: count, dtype: int64

Promo2 start years:
Promo2SinceYear
2009.0     73
2010.0     64
2011.0    128
2012.0     81
2013.0    120
2014.0     95
2015.0     10
Name: count, dtype: int64

Number of distinct (year, week) start points: 55


There are 544 stores that did not participate, they will be our control group for the DiD. Now we will see what stores participate in the time window, filtering out stores that were always participating and ones that go past the data time window.

In [43]:
import numpy as np

s = store.copy()

# Convert ISO year + week into an actual date (Monday of that week)
s['Promo2Start'] = pd.to_datetime(
    s['Promo2SinceYear'].astype('Int64').astype(str) + '-' +
    s['Promo2SinceWeek'].astype('Int64').astype(str) + '-1',
    format='%Y-%W-%w',
    errors='coerce'
)

data_start = df['Date'].min()
data_end   = df['Date'].max()

# Classify every store into one of three groups
def classify(row):
    if row['Promo2'] == 0:
        return 'never_treated'
    if pd.isna(row['Promo2Start']):
        return 'unknown'
    if row['Promo2Start'] < data_start:
        return 'always_treated'   # joined before our window - no pre-period
    if row['Promo2Start'] > data_end:
        return 'post_window'
    return 'treated_in_window'

s['group'] = s.apply(classify, axis=1)
print(s['group'].value_counts())
print("Treated-in-window start dates range:",
      s.loc[s['group']=='treated_in_window', 'Promo2Start'].min(), "to",
      s.loc[s['group']=='treated_in_window', 'Promo2Start'].max())

group
never_treated        544
always_treated       346
treated_in_window    225
Name: count, dtype: int64
Treated-in-window start dates range: 2013-01-07 00:00:00 to 2015-06-08 00:00:00


We see that 225 stores particpated in the time window, that's a total of 769 stores for our data, we will drop the stores that have always had the promo or the promo past the data window

In [44]:
# Bring group label and treatment start date onto the main panel
df = df.merge(
    s[['Store', 'group', 'Promo2Start']],
    on='Store',
    how='left'
)

# Drop always-treated stores - no observable pre-period
analysis = df[df['group'].isin(['never_treated', 'treated_in_window'])].copy()

print("Rows before:", len(df))
print("Rows after dropping always_treated:", len(analysis))

print(analysis['group'].value_counts())

print("Stores in analysis sample:", analysis['Store'].nunique())

Rows before: 1017209
Rows after dropping always_treated: 707469
group
never_treated        508031
treated_in_window    199438
Name: count, dtype: int64
Stores in analysis sample: 769
